# DistilBERT Training — Google Colab

Run this notebook on Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU).
Expected time: ~10–15 minutes.

**After training:** download `distilbert.tar.gz` from the Files panel and unpack it into `models/distilbert/` in the repo.

In [ ]:
# AI-assisted (Claude Code, claude.ai) — https://claude.ai

# Verify GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected — check Runtime type')

In [ ]:
%pip install -q transformers datasets scikit-learn pandas

In [ ]:
# Upload features.csv from your local repo (data/processed/features.csv)
from google.colab import files
uploaded = files.upload()  # select features.csv

In [ ]:
import pandas as pd

df = pd.read_csv('features.csv')
print(f'Total rows: {len(df)}')
print('Grade distribution:')
print(df['grade'].value_counts())

rows_with_text = df['combined_reviews'].fillna('').str.strip().ne('').sum()
print(f'\nRows with review text: {rows_with_text} ({rows_with_text/len(df):.1%})')
print('Note: empty reviews are kept — model trains on all rows for class diversity')

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2

texts = df['combined_reviews'].fillna('').tolist()
labels = df['grade_encoded'].tolist()
num_labels = len(set(labels))

texts_train, texts_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)

print(f'Train: {len(texts_train)}  Test: {len(texts_test)}  Classes: {num_labels}')

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import numpy as np
from sklearn.metrics import classification_report


class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_enc = tokenizer(texts_train, truncation=True, padding=True, max_length=256)
test_enc  = tokenizer(texts_test,  truncation=True, padding=True, max_length=256)

train_dataset = ReviewDataset(train_enc, y_train)
test_dataset  = ReviewDataset(test_enc,  y_test)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3  # A=0, B=1, C=2
)

training_args = TrainingArguments(
    output_dir='./distilbert',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_dir='./distilbert_logs',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

In [ ]:
# Evaluate on test set
preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)

grade_names = {0: 'A', 1: 'B', 2: 'C'}
present_labels = sorted(set(y_test))
present_names = [grade_names[l] for l in present_labels]
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names))

In [ ]:
# Save model + tokenizer, then download as a tarball
import os

SAVE_DIR = './distilbert_final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

os.system(f'tar -czf distilbert.tar.gz {SAVE_DIR}')
files.download('distilbert.tar.gz')
print('Download started — unpack into models/distilbert/ in the repo')